<a href="https://colab.research.google.com/github/Lee-Minsoo-97/Sales-Data-Prediction/blob/gemini_original/iHerb_Sales_Pred_ML_Project_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error
from google.colab import drive

# --- 1. Google Drive 마운트 ---
drive.mount('/content/drive')

# --- 2. 모델링용 데이터 불러오기 및 분리 ---
file_path = '/content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv'
df_model = pd.read_csv(file_path, parse_dates=['Date'])
print("✅ 모델링용 데이터를 성공적으로 불러왔습니다.")

# ★★★ 누락된 부분 추가: X_test, y_test 생성 ★★★
# 훈련/테스트 데이터 분리
test_start_date = '2025-07-01'
test_df = df_model[df_model['Date'] >= test_start_date]

# X_test, y_test 정의
cols_to_drop = ['Date', 'SKU', 'UPC Code', 'Product Description', 'Sales']
X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df['Sales']
print("✅ 테스트 데이터(X_test, y_test) 준비 완료.")
# --------------------------------------------------

# --- 3. 저장된 최종 모델들 불러오기 ---
lgbm_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/lgbm_model_v1_1.pkl'
lgbm_model = joblib.load(lgbm_model_filename)

xgb_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/xgb_model_v2_0.pkl'
xgb_model = joblib.load(xgb_model_filename)
print("✅ v1.1 (LightGBM)과 v2.0 (XGBoost) 모델을 성공적으로 불러왔습니다.")

# --- 4. 앙상블 예측 및 성능 평가 ---
print("\n🚀 각 모델의 예측을 수행합니다...")
lgbm_preds = lgbm_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

ensemble_preds = 0.5 * lgbm_preds + 0.5 * xgb_preds
ensemble_mae = mean_absolute_error(y_test, ensemble_preds)

print("\n--- 최종 모델별 예측 성능 비교 ---")
print(f"LightGBM v1.1 최종 MAE: {mean_absolute_error(y_test, lgbm_preds):.2f}")
print(f"XGBoost v2.0 최종 MAE:  {mean_absolute_error(y_test, xgb_preds):.2f}")
print(f"🚀 앙상블 최종 MAE:        {ensemble_mae:.2f}")

results_df_ensemble = pd.DataFrame({
    'Date': test_df['Date'],
    'SKU': test_df['SKU'],
    'Actual_Sales': y_test,
    'Ensemble_Preds': ensemble_preds
})
print("\n--- 실제값 vs 앙상블 예측값 샘플 ---")
print(results_df_ensemble.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 모델링용 데이터를 성공적으로 불러왔습니다.
✅ 테스트 데이터(X_test, y_test) 준비 완료.
✅ v1.1 (LightGBM)과 v2.0 (XGBoost) 모델을 성공적으로 불러왔습니다.

🚀 각 모델의 예측을 수행합니다...

--- 최종 모델별 예측 성능 비교 ---
LightGBM v1.1 최종 MAE: 32.13
XGBoost v2.0 최종 MAE:  35.79
🚀 앙상블 최종 MAE:        33.27

--- 실제값 vs 앙상블 예측값 샘플 ---
         Date       SKU  Actual_Sales  Ensemble_Preds
12 2025-07-01  APB68267           949     1014.307068
13 2025-08-01  APB68267          1130      804.688858
26 2025-07-01  APB68268          1233     1116.877435
27 2025-08-01  APB68268          1164     1190.534538
47 2025-07-01  APB68270           435      964.495184


/usr/lib/python3.12/pickle.py:1760: UserWarning: [20:54:49] WARNING: /workspace/src/gbm/gbtree.cc:384: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [20:54:49] WARNING: /workspace/src/context.cc:49: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [20:54:49] WARNING: /workspace/src/context.cc:203: XGBoost is not compiled with CUDA support.
  setstate(state)


In [ ]:
results_df_ensemble

,Date,SKU,Actual_Sales,Ensemble_Preds
12,2025-07-01,APB68267,949,1014.307068
13,2025-08-01,APB68267,1130,804.688858
26,2025-07-01,APB68268,1233,1116.877435
27,2025-08-01,APB68268,1164,1190.534538
47,2025-07-01,APB68270,435,964.495184
...,...,...,...,...
4638,2025-08-01,TWA31353,8,10.257792
4640,2025-07-01,TWA31354,1,7.846502
4641,2025-08-01,TWA31354,9,10.215067
4643,2025-07-01,TWA60548,0,0.172132


In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error
from google.colab import drive

# --- 1. 데이터 및 모델 불러오기 ---
# Google Drive 마운트
drive.mount('/content/drive')

# 모델링용 데이터 불러오기
file_path = '/content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv'
df_model = pd.read_csv(file_path, parse_dates=['Date'])

# 테스트 데이터(X_test, y_test) 생성
test_start_date = '2025-07-01'
test_df = df_model[df_model['Date'] >= test_start_date]
cols_to_drop = ['Date', 'SKU', 'UPC Code', 'Product Description', 'Sales']
X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df['Sales']
print("✅ 테스트 데이터를 준비했습니다.")

# v1.2 LightGBM 모델 불러오기
lgbm_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/lgbm_model_v1.2.pkl'
lgbm_model = joblib.load(lgbm_model_filename)

# v2.0 XGBoost 모델 불러오기
xgb_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/xgb_model_v2_0.pkl'
xgb_model = joblib.load(xgb_model_filename)

print("✅ v1.2 (LightGBM)과 v2.0 (XGBoost) 모델을 성공적으로 불러왔습니다.")


# --- 2. 앙상블 예측 및 성능 평가 ---
print("\n🚀 각 모델의 예측을 수행합니다...")
lgbm_preds = lgbm_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

# 50:50 가중 평균으로 앙상블
ensemble_preds = 0.5 * lgbm_preds + 0.5 * xgb_preds

# 각 모델 및 앙상블 모델의 최종 MAE 점수 계산
mae_lgbm = mean_absolute_error(y_test, lgbm_preds)
mae_xgb = mean_absolute_error(y_test, xgb_preds)
mae_ensemble = mean_absolute_error(y_test, ensemble_preds)

print("\n--- 최종 모델별 예측 성능 비교 ---")
print(f"LightGBM v1.2 최종 MAE: {mae_lgbm:.2f}")
print(f"XGBoost v2.0 최종 MAE:  {mae_xgb:.2f}")
print(f"🚀 앙상블(50:50) 최종 MAE: {mae_ensemble:.2f}")

# --- 3. (보너스) 가중 평균 앙상블 ---
# 각 모델의 교차 검증 점수를 기반으로 가중치 부여 (더 잘한 모델에 높은 가중치)
# LGBM CV MAE: 28.67, XGB CV MAE: 27.03
# 가중치는 점수가 낮을수록(더 좋을수록) 높아야 하므로, 점수의 역수를 사용합니다.
inv_mae_lgbm = 1 / 28.67
inv_mae_xgb = 1 / 27.03
total_inv_mae = inv_mae_lgbm + inv_mae_xgb

weight_lgbm = inv_mae_lgbm / total_inv_mae
weight_xgb = inv_mae_xgb / total_inv_mae

weighted_ensemble_preds = weight_lgbm * lgbm_preds + weight_xgb * xgb_preds
weighted_ensemble_mae = mean_absolute_error(y_test, weighted_ensemble_preds)

print(f"\n--- 가중 평균 앙상블 (LGBM {weight_lgbm:.0%}, XGB {weight_xgb:.0%}) ---")
print(f"🚀 가중 앙상블 최종 MAE:   {weighted_ensemble_mae:.2f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 테스트 데이터를 준비했습니다.
✅ v1.2 (LightGBM)과 v2.0 (XGBoost) 모델을 성공적으로 불러왔습니다.

🚀 각 모델의 예측을 수행합니다...

--- 최종 모델별 예측 성능 비교 ---
LightGBM v1.2 최종 MAE: 40.81
XGBoost v2.0 최종 MAE:  35.79
🚀 앙상블(50:50) 최종 MAE: 38.08

--- 가중 평균 앙상블 (LGBM 49%, XGB 51%) ---
🚀 가중 앙상블 최종 MAE:   38.00


/usr/lib/python3.12/pickle.py:1760: UserWarning: [21:20:45] WARNING: /workspace/src/gbm/gbtree.cc:384: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [21:20:45] WARNING: /workspace/src/context.cc:49: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [21:20:45] WARNING: /workspace/src/context.cc:203: XGBoost is not compiled with CUDA support.
  setstate(state)


In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error
from google.colab import drive

# --- 1. 데이터 및 모델 불러오기 ---
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Colab Notebooks/Data for Modeling/df_for_modeling.csv'
df_model = pd.read_csv(file_path, parse_dates=['Date'])

test_start_date = '2025-07-01'
test_df = df_model[df_model['Date'] >= test_start_date]
cols_to_drop = ['Date', 'SKU', 'UPC Code', 'Product Description', 'Sales']
X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df['Sales']

lgbm_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/lgbm_model_v1.2.pkl'
lgbm_model = joblib.load(lgbm_model_filename)
xgb_model_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/xgb_model_v2_0.pkl'
xgb_model = joblib.load(xgb_model_filename)

# --- 2. 예측 수행 ---
lgbm_preds = lgbm_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

# --- 3. 앙상블 예측 생성 ---
# 50:50 앙상블
ensemble_preds = 0.5 * lgbm_preds + 0.5 * xgb_preds

# 가중 평균 앙상블
inv_mae_lgbm = 1 / 28.67
inv_mae_xgb = 1 / 27.03
total_inv_mae = inv_mae_lgbm + inv_mae_xgb
weight_lgbm = inv_mae_lgbm / total_inv_mae
weight_xgb = inv_mae_xgb / total_inv_mae
weighted_ensemble_preds = weight_lgbm * lgbm_preds + weight_xgb * xgb_preds

# --- 4. 최종 결과 데이터프레임 생성 ---
final_results_df = pd.DataFrame({
    'Date': test_df['Date'],
    'SKU': test_df['SKU'],
    'Actual_Sales': y_test,
    'Prediction_LGBM_v1.2': lgbm_preds,
    'Prediction_XGB_v2.0': xgb_preds,
    'Prediction_Ensemble_50_50': ensemble_preds,
    'Prediction_Ensemble_Weighted': weighted_ensemble_preds
})

# 소수점 둘째 자리까지 반올림
final_results_df = final_results_df.round(2)

# --- 5. CSV 파일로 저장 ---
output_filename = '/content/drive/MyDrive/Colab Notebooks/Trained Models/final_predictions.csv'

# index=False 옵션으로 불필요한 인덱스 컬럼 저장을 방지
final_results_df.to_csv(output_filename, index=False)

print(f"✅ 예측 결과가 CSV 파일로 저장되었습니다.")
print(f"   경로: {output_filename}")

print("\n--- 저장된 파일 샘플 데이터 ---")
print(final_results_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 예측 결과가 CSV 파일로 저장되었습니다.
   경로: /content/drive/MyDrive/Colab Notebooks/Trained Models/final_predictions.csv

--- 저장된 파일 샘플 데이터 ---
         Date       SKU  Actual_Sales  Prediction_LGBM_v1.2  \
12 2025-07-01  APB68267           949                580.64   
13 2025-08-01  APB68267          1130                360.15   
26 2025-07-01  APB68268          1233                583.79   
27 2025-08-01  APB68268          1164                607.45   
47 2025-07-01  APB68270           435                578.00   

    Prediction_XGB_v2.0  Prediction_Ensemble_50_50  \
12           920.190002                     750.42   
13           682.619995                     521.38   
26          1011.130005                     797.46   
27          1046.050049                     826.75   
47           804.400024                     691.20   

    Prediction_Ensemble_Weighted  


/usr/lib/python3.12/pickle.py:1760: UserWarning: [21:23:20] WARNING: /workspace/src/gbm/gbtree.cc:384: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [21:23:20] WARNING: /workspace/src/context.cc:49: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1760: UserWarning: [21:23:20] WARNING: /workspace/src/context.cc:203: XGBoost is not compiled with CUDA support.
  setstate(state)
